# TimesFM - Foundation Time Series Model (Bonus)
imports, set up mlflow.

In [7]:
import time

import numpy as np
import pandas as pd
import mlflow
import timesfm

from preprocessing import load_raw, build_features, wmae

np.random.seed(42)
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # relative path avoids an MLflow bug
                                                 # that mis-encodes spaces in Windows paths
mlflow.set_experiment("TimesFM_Training")

REPO_ID = "google/timesfm-2.5-200m-pytorch"

## 1. Load & Reshape Data

In [8]:
train, test, features, stores = load_raw()
train_full = build_features(train, features, stores)

long_df = train_full[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]].rename(
    columns={"Date": "ds", "Weekly_Sales": "y"}
)
long_df["unique_id"] = long_df["Store"].astype(str) + "_" + long_df["Dept"].astype(str)
long_df = long_df[["unique_id", "ds", "y", "IsHoliday"]].sort_values(["unique_id", "ds"])

print(long_df.shape)
long_df.head()

(421570, 4)


,unique_id,ds,y,IsHoliday
87524,10_1,2010-02-05,40212.84,False
87525,10_1,2010-02-12,67699.32,True
87526,10_1,2010-02-19,49748.33,False
87527,10_1,2010-02-26,33601.22,False
87528,10_1,2010-03-05,36572.44,False


## 2. Filter Series

In [9]:
H = 39
INPUT_SIZE = 52
MIN_LEN = INPUT_SIZE + H

series_len = long_df.groupby("unique_id").size()
keep_ids = series_len[series_len >= MIN_LEN].index

print(f"series total: {series_len.shape[0]}")
print(f"series kept (len >= {MIN_LEN}): {len(keep_ids)}")
print(f"series dropped (too short): {series_len.shape[0] - len(keep_ids)}")

long_df = long_df[long_df["unique_id"].isin(keep_ids)].reset_index(drop=True)
print("rows after filtering:", long_df.shape[0])

with mlflow.start_run(run_name="TimesFM_Preprocessing"):
    mlflow.log_param("horizon_h", H)
    mlflow.log_param("input_size", INPUT_SIZE)
    mlflow.log_param("min_series_length", MIN_LEN)
    mlflow.log_metric("series_total", series_len.shape[0])
    mlflow.log_metric("series_kept", len(keep_ids))
    mlflow.log_metric("series_dropped", series_len.shape[0] - len(keep_ids))
    mlflow.log_metric("rows_kept", long_df.shape[0])

series total: 3331
series kept (len >= 91): 2900
series dropped (too short): 431
rows after filtering: 410069


## 3. Zero-Shot Holdout Evaluation

There's no `neuralforecast`-style `cross_validation()` here since TimesFM isn't part of that library and — more importantly — there's nothing to fit. Instead we build the same time-based holdout manually: for every kept series, the last `H` weeks are the holdout target and the `INPUT_SIZE` weeks immediately before that are the context TimesFM sees. The model never observes any Walmart data before making each forecast — it's applying patterns learned from its (undisclosed, general-purpose) pretraining corpus directly to this task.

`from_pretrained()` downloads the checkpoint from Hugging Face Hub on first use (~130s, one-time) and caches it locally afterward (~2s to reload). **This requires network access** — a real operational difference from the other four notebooks, which only need local files once training is done.

In [ ]:
contexts, holds = [], []
for uid, g in long_df.groupby("unique_id"):
    g = g.sort_values("ds")
    ctx, hold = g.iloc[-(H + INPUT_SIZE):-H], g.iloc[-H:]
    if len(ctx) == INPUT_SIZE and len(hold) == H:
        contexts.append(ctx["y"].values.astype(np.float32))
        holds.append(hold)

print(f"series with a full context+holdout window: {len(contexts)}")

t0 = time.time()
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(REPO_ID)
load_seconds = time.time() - t0
print(f"model load took {load_seconds:.1f} sec")

model.compile(timesfm.ForecastConfig(
    max_context=INPUT_SIZE,
    max_horizon=H,
    normalize_inputs=True,
    use_continuous_quantile_head=False,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
))

t0 = time.time()
point, quantiles = model.forecast(horizon=H, inputs=contexts)
forecast_seconds = time.time() - t0
print(f"zero-shot forecast for {len(contexts)} series took {forecast_seconds:.1f} sec")

series with a full context+holdout window: 2900
model load took 3.2 sec


In [ ]:
hold_df = pd.concat(holds).reset_index(drop=True)
hold_df["TimesFM"] = point.reshape(-1)

val_wmae = wmae(hold_df["y"], hold_df["TimesFM"], hold_df["IsHoliday"])
val_mae = (hold_df["y"] - hold_df["TimesFM"]).abs().mean()

print(f"Validation WMAE: {val_wmae:.2f}")
print(f"Validation MAE:  {val_mae:.2f}")

with mlflow.start_run(run_name="TimesFM_ZeroShot_CV"):
    mlflow.log_param("repo_id", REPO_ID)
    mlflow.log_param("horizon_h", H)
    mlflow.log_param("input_size", INPUT_SIZE)
    mlflow.log_param("fine_tuned", False)
    mlflow.log_metric("model_load_seconds", load_seconds)
    mlflow.log_metric("forecast_seconds", forecast_seconds)
    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("val_mae", val_mae)

Validation WMAE: 2507.66
Validation MAE:  2429.92


## 4. Pipeline Export

No local weights to save here — the "model" is just a reference to the frozen pretrained checkpoint (`REPO_ID`) plus the same `ForecastConfig` validated above. The pipeline's `predict()` still has to satisfy the "runs on raw, unprocessed test data" requirement, but the challenge is different from TFT's: raw `test.csv` has no historical `Weekly_Sales` at all (it's purely future dates to predict), so `predict()` loads `train.csv` internally via `load_raw()` to reconstruct each requested Store/Dept's last `INPUT_SIZE` weeks of context before calling the frozen model. `code_paths=["preprocessing.py"]` bundles `preprocessing.py` into the logged artifact.

This assumes every series' test-set forecast horizon starts immediately after its last `train.csv` date and spans exactly `H` weeks at a `W-FRI` cadence — true for this competition (`train.csv` ends 2012-10-26, `test.csv` starts 2012-11-02 for every Store/Dept), but worth re-verifying against the real file before trusting `model_inference.ipynb` output (see notes).

In [ ]:
class TimesFMPipeline(mlflow.pyfunc.PythonModel):
    """Wraps the frozen, pretrained TimesFM checkpoint to run on raw
    Store/Dept/Date input (e.g. Kaggle test.csv), unprocessed. Unlike the other
    four pipelines, nothing was fit to this dataset: predict() reconstructs each
    series' recent history from train.csv and forecasts zero-shot."""

    REPO_ID = "google/timesfm-2.5-200m-pytorch"
    INPUT_SIZE = 52
    HORIZON = 39

    def load_context(self, context):
        import timesfm

        self.model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(self.REPO_ID)
        self.model.compile(timesfm.ForecastConfig(
            max_context=self.INPUT_SIZE,
            max_horizon=self.HORIZON,
            normalize_inputs=True,
            use_continuous_quantile_head=False,
            force_flip_invariance=True,
            infer_is_positive=True,
            fix_quantile_crossing=True,
        ))

    def predict(self, context, model_input: pd.DataFrame):
        import numpy as np
        import pandas as pd
        from preprocessing import load_raw

        train, _, _, _ = load_raw()
        train = train.copy()
        train["unique_id"] = train["Store"].astype(str) + "_" + train["Dept"].astype(str)

        df = model_input.copy()
        df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
        df["ds"] = pd.to_datetime(df["Date"])

        contexts, valid_uids, last_dates = [], [], []
        for uid in df["unique_id"].unique():
            hist = train[train["unique_id"] == uid].sort_values("Date")
            if len(hist) >= self.INPUT_SIZE:
                contexts.append(hist["Weekly_Sales"].values[-self.INPUT_SIZE:].astype("float32"))
                valid_uids.append(uid)
                last_dates.append(hist["Date"].max())

        preds_df = pd.DataFrame(columns=["unique_id", "ds", "Weekly_Sales"])
        if contexts:
            point, _ = self.model.forecast(horizon=self.HORIZON, inputs=contexts)
            rows = []
            for uid, last_date, series_point in zip(valid_uids, last_dates, point):
                fcst_dates = pd.date_range(
                    last_date + pd.Timedelta(weeks=1), periods=self.HORIZON, freq="W-FRI"
                )
                rows.append(pd.DataFrame({
                    "unique_id": uid, "ds": fcst_dates, "Weekly_Sales": series_point,
                }))
            preds_df = pd.concat(rows, ignore_index=True)

        out = df[["Store", "Dept", "Date", "unique_id", "ds"]].merge(
            preds_df, on=["unique_id", "ds"], how="left"
        )
        return out[["Store", "Dept", "Date", "Weekly_Sales"]]


with mlflow.start_run(run_name="TimesFM_Final") as run:
    mlflow.log_param("repo_id", REPO_ID)
    mlflow.log_param("horizon_h", H)
    mlflow.log_param("input_size", INPUT_SIZE)
    mlflow.log_param("fine_tuned", False)
    mlflow.log_metric("val_wmae", val_wmae)  # carried over from the zero-shot CV run above

    mlflow.pyfunc.log_model(
        artifact_path="timesfm_pipeline",
        python_model=TimesFMPipeline(),
        code_paths=["preprocessing.py"],
    )
    print("Logged pipeline under run:", run.info.run_id)

2026/07/11 22:31:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 22:31:19 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting
2026/07/11 22:31:19 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/11 22:31:19 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
2026/07/11 22:31:19 INFO mlflow.utils.uv_utils: Detected uv project: found uv

Logged pipeline under run: 2891daa8bd024e19883773b01d08549c
